# 3부 결정 트리, 배깅, 그리고 랜덤 포레스트 — 학생 노트북

각 절은 **Ping**(교수자 시범 — 그대로 실행해 결과를 본다)과 **Pong**(학생 실습 — `____` 빈칸을 채워 실행한다)으로 이어진다. Pong의 정답은 바로 아래 토글에 있다. 먼저 스스로 채운 뒤 펼쳐 확인한다. 데이터는 1부의 표준 전처리를 그대로 쓴다.

## 환경 준비 (한 번만 실행)

In [ ]:
# !pip install pandas numpy matplotlib scikit-learn koreanize-matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib   # 한글 폰트(NanumGothic) 자동 설정

URL = "https://raw.githubusercontent.com/leina99-lab/classes/main/AI%ED%94%84%EB%A1%9C%EA%B7%B8%EB%9E%98%EB%B0%8D/data/AmesHousing.csv"
df_raw = pd.read_csv(URL)

def prepare_ames(df_in):
    df = df_in.copy()
    df = df.drop(columns=[c for c in ["Order", "PID"] if c in df.columns])
    none_cols = ["Pool QC", "Misc Feature", "Alley", "Fence", "Fireplace Qu",
                 "Garage Qual", "Garage Cond", "Garage Finish", "Garage Type",
                 "Bsmt Qual", "Bsmt Cond", "Bsmt Exposure",
                 "BsmtFin Type 1", "BsmtFin Type 2", "Mas Vnr Type"]
    for c in none_cols:
        if c in df.columns:
            df[c] = df[c].fillna("None")
    num_cols = df.select_dtypes("number").columns
    df[num_cols] = df[num_cols].fillna(df[num_cols].median())
    if "Gr Liv Area" in df.columns:
        df = df[df["Gr Liv Area"] < 4000].copy()
    if all(c in df.columns for c in ["1st Flr SF", "2nd Flr SF", "Total Bsmt SF"]):
        df["Total SF"] = df["1st Flr SF"] + df["2nd Flr SF"] + df["Total Bsmt SF"]
    return df

df = prepare_ames(df_raw)
y = np.log1p(df["SalePrice"])
X = df.select_dtypes("number").drop(columns=["SalePrice"])
print(f"준비 완료: {X.shape[0]}채 × {X.shape[1]}개 변수")

---

## 0장 결정 트리 — 질문의 사슬로 답을 찾는 모델

결정 트리는 예/아니오 질문을 차례로 던져 잎 노드에 도달하면 그 자리의 값을 답으로 낸다. 회귀에서는 노드 안 가격들의 흩어짐(분산, MSE)을 가장 많이 줄이는 질문을 고르고, 잎의 예측값은 그 노드 학습 가격들의 평균이다.

![결정 트리](https://raw.githubusercontent.com/leina99-lab/classes/main/AI%ED%94%84%EB%A1%9C%EA%B7%B8%EB%9E%98%EB%B0%8D/decesion_tree/figs/01_decision_tree_basic.png)

### Ping 1 — 깊이 무제한 트리는 데이터를 외운다

In [ ]:
from sklearn.tree import DecisionTreeRegressor

tree = DecisionTreeRegressor(random_state=42)
tree.fit(X, y)
print(f"트리 깊이:  {tree.get_depth()}")
print(f"잎 노드 수: {tree.get_n_leaves()}")
print(f"학습 R²:    {tree.score(X, y):.4f}")   # 거의 1.0 → 과적합 신호

### Pong 1 — 깊이를 바꿔 과적합이 시작되는 지점 찾기

In [ ]:
from sklearn.model_selection import cross_val_score

for d in [1, 3, 5, 8, 12, 20, None]:
    t = DecisionTreeRegressor(max_depth=____, random_state=42)   # ① 깊이 d
    t.fit(X, y)
    train_r2 = t.score(X, y)
    cv_r2 = cross_val_score(t, X, y, cv=5, scoring="r2").____()  # ② 평균
    print(f"depth={str(d):>5}  train={train_r2:.4f}  cv={cv_r2:.4f}")

<details><summary>▶ 정답 보기</summary>

① `d` &nbsp;&nbsp; ② `mean`

검증 R²가 다시 떨어지기 시작하는 깊이 6~10 근처가 단일 트리의 황금 지점이다.
</details>

---

## 1장 한 그루 트리의 흔들림과 평균의 안정

한 그루 트리는 학습 데이터에 매우 민감하다(분산이 크다). 서로 조금씩 다른 트리를 여러 그루 평균내면 우연이 상쇄되어 분산이 줄어든다.

### Ping 2 — 데이터 30%만 빠져도 첫 질문이 바뀐다

In [ ]:
from collections import Counter

rng = np.random.default_rng(42)
n = len(X)
first_splits = []
for seed in range(20):
    idx = rng.choice(n, size=int(n * 0.7), replace=False)
    t = DecisionTreeRegressor(max_depth=3, random_state=42)
    t.fit(X.iloc[idx], y.iloc[idx])
    first_splits.append(X.columns[t.tree_.feature[0]])

for var, cnt in Counter(first_splits).most_common():
    print(f"{var:<20s} {cnt}회")

### Pong 2 — 표본 비율을 80%로 바꾸면 흔들림이 줄어드는지 보기

In [ ]:
rng = np.random.default_rng(0)
first_splits = []
for seed in range(20):
    idx = rng.choice(n, size=int(n * ____), replace=False)   # ① 0.8 (80% 표본)
    t = DecisionTreeRegressor(max_depth=3, random_state=42)
    t.fit(X.iloc[idx], y.iloc[idx])
    first_splits.append(X.columns[t.tree_.feature[____]])    # ② 뿌리 노드 0
print(Counter(first_splits).most_common())

<details><summary>▶ 정답 보기</summary>

① `0.8` &nbsp;&nbsp; ② `0`

표본을 많이 쓸수록 첫 질문이 `Overall Qual`로 더 자주 고정된다 — 흔들림(분산)이 준다.
</details>

---

## 2장 복원 추출과 부트스트랩 표본

복원 추출로 원본과 같은 크기의 표본을 뽑으면 어떤 행은 여러 번, 어떤 행은 한 번도 안 들어간다. 안 들어간 약 36.8%가 OOB 표본이다. 부트스트랩 표본마다 트리를 학습시켜 예측을 평균내는 것이 배깅이다.

![배깅 다이어그램](https://raw.githubusercontent.com/leina99-lab/classes/main/AI%ED%94%84%EB%A1%9C%EA%B7%B8%EB%9E%98%EB%B0%8D/decesion_tree/figs/04_bagging_diagram.png)

### Ping 3 — 부트스트랩의 63.2% / 36.8% 확인

In [ ]:
rng = np.random.default_rng(42)
n = len(X)
boot_idx = rng.choice(n, size=n, replace=True)        # 복원 추출
unique_idx = np.unique(boot_idx)
oob_idx = np.setdiff1d(np.arange(n), unique_idx)      # 안 뽑힌 행

print(f"고유 행: {len(unique_idx)/n*100:.1f}%")   # 약 63.2
print(f"OOB 행: {len(oob_idx)/n*100:.1f}%")       # 약 36.8

### Pong 3 — 부트스트랩 트리를 OOB로 평가하기

In [ ]:
from sklearn.metrics import r2_score

rng = np.random.default_rng(42)
for b in range(5):
    boot_idx = rng.choice(n, size=n, replace=____)               # ① 복원 추출 True
    oob_mask = np.setdiff1d(np.arange(n), np.unique(boot_idx))
    t = DecisionTreeRegressor(max_depth=10, random_state=42)
    t.fit(X.iloc[boot_idx], y.iloc[boot_idx])
    oob_pred = t.predict(X.iloc[oob_mask])
    print(f"트리 {b+1}: OOB R² = {r2_score(y.iloc[oob_mask], oob_pred):.4f}")

<details><summary>▶ 정답 보기</summary>

① `True`

5그루의 OOB R²가 0.6~0.7 사이에서 흔들린다 — 이것이 단일 트리의 분산이다.
</details>

---

## 3장 평균의 한계 — 트리들이 닮으면

부트스트랩 트리들은 같은 원본에서 나와 서로 닮아 있어, 분산이 0이 아니라 어떤 바닥까지만 줄어든다. 트리 수를 늘리면 수확체감이 일어난다.

![분산 감소](https://raw.githubusercontent.com/leina99-lab/classes/main/AI%ED%94%84%EB%A1%9C%EA%B7%B8%EB%9E%98%EB%B0%8D/decesion_tree/figs/05_variance_reduction.png)

### Ping 4 — 트리 수에 따른 R² 곡선(수확체감)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

xs = [1, 2, 5, 10, 20, 50, 100, 200]
for nt in xs:
    rf = RandomForestRegressor(n_estimators=nt, random_state=42, n_jobs=-1)
    r2 = cross_val_score(rf, X, y, cv=3, scoring="r2", n_jobs=-1).mean()
    print(f"{nt:>4d}그루: R² = {r2:.4f}")

### Pong 4 — 곡선을 그려 평탄해지는 지점 확인하기

In [ ]:
scores = []
xs = [1, 2, 5, 10, 20, 50, 100, 200]
for nt in xs:
    rf = RandomForestRegressor(n_estimators=____, random_state=42, n_jobs=-1)  # ① nt
    scores.append(cross_val_score(rf, X, y, cv=3, scoring="r2", n_jobs=-1).____())  # ② mean
plt.plot(xs, scores, "-o"); plt.xlabel("트리 수"); plt.ylabel("R²")
plt.title("트리 수에 따른 분산 감소(수확체감)"); plt.show()

<details><summary>▶ 정답 보기</summary>

① `nt` &nbsp;&nbsp; ② `mean`

약 50그루 근처에서 곡선이 평탄해진다.
</details>

---

## 4장 트리들을 덜 닮게 — 무작위 변수 선택

강한 변수 `Overall Qual`(가격과 상관 0.80)이 거의 모든 트리의 첫 질문이 되어 트리들이 닮는다. 매 분할마다 변수의 일부만 후보로 두면(`max_features`) 닮음이 깨진다. 회귀 권장값은 전체의 약 1/3이다.

![mtry 줄다리기](https://raw.githubusercontent.com/leina99-lab/classes/main/AI%ED%94%84%EB%A1%9C%EA%B7%B8%EB%9E%98%EB%B0%8D/decesion_tree/figs/06_mtry_tradeoff.png)

### Ping 5 — max_features를 바꿔 가며 비교

In [ ]:
p = X.shape[1]
for mf in [3, 5, p // 4, p // 3, p // 2, p]:
    rf = RandomForestRegressor(n_estimators=100, max_features=mf,
                               random_state=42, n_jobs=-1)
    r2 = cross_val_score(rf, X, y, cv=3, scoring="r2", n_jobs=-1).mean()
    print(f"max_features={mf:>3d}: R² = {r2:.4f}")

### Pong 5 — 회귀 권장값(p/3)으로 숲 만들기

In [ ]:
p = X.shape[1]
rf = RandomForestRegressor(n_estimators=200,
                           max_features=p // ____,   # ① 회귀 권장: 전체의 1/3
                           random_state=42, n_jobs=-1)
rf.fit(X, y)
print("학습 R²:", rf.score(X, y).round(4))

<details><summary>▶ 정답 보기</summary>

① `3`

`max_features = p/3` 근처에서 앙상블 성능이 최대가 된다.
</details>

---

## 5장 검증 데이터 없이 점수 매기기 — OOB 점수

한 행을 OOB로 가진 트리(평균 약 37그루)의 예측 평균이 그 행의 일반화 예측이다. 모든 데이터를 학습에 쓰면서 한 번의 학습으로 검증까지 끝나며, 5-fold CV와 거의 같고 약 5배 빠르다.

### Ping 6 — OOB 점수 vs 5-fold CV

In [ ]:
import time
rf = RandomForestRegressor(n_estimators=100, oob_score=True,
                           random_state=42, n_jobs=-1)
t0 = time.time(); rf.fit(X, y); oob_t = time.time() - t0
print(f"OOB R²: {rf.oob_score_:.4f}  ({oob_t:.2f}s)")

t0 = time.time()
cv = cross_val_score(rf, X, y, cv=5, scoring="r2", n_jobs=-1).mean()
print(f"CV  R²: {cv:.4f}  ({time.time()-t0:.2f}s)")

### Pong 6 — OOB 점수만 한 줄로 얻기

In [ ]:
rf = RandomForestRegressor(n_estimators=200, ____=True,   # ① OOB 평가 켜기
                           random_state=42, n_jobs=-1)
rf.fit(X, y)
print("OOB R²:", rf.____)                                # ② OOB 점수 속성

<details><summary>▶ 정답 보기</summary>

① `oob_score` &nbsp;&nbsp; ② `oob_score_`
</details>

---

## 6장 어떤 변수가 중요한가 — 두 가지 답

MDI는 변수가 *얼마나 자주 쓰였는지*(`feature_importances_`), 순열 중요도는 변수가 *없어도 되는지*를 묻는다. 사촌 변수 `Garage Cars`·`Garage Area`(상관 0.89)는 MDI에선 둘 다 적당히, 순열에선 둘 다 거의 0을 받는다.

![중요도 비교](https://raw.githubusercontent.com/leina99-lab/classes/main/AI%ED%94%84%EB%A1%9C%EA%B7%B8%EB%9E%98%EB%B0%8D/decesion_tree/figs/07_importance_compare.png)

### Ping 7 — MDI 변수 중요도

In [ ]:
rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X, y)
mdi = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print(mdi.head(8))   # Overall Qual 1위, Total SF 상위권

### Pong 7 — 사촌 변수의 대체 가능성 확인

In [ ]:
for name, drop in [("전체", []), ("Cars만 제거", ["Garage Cars"]),
                   ("둘 다 제거", ["Garage Cars", "Garage Area"])]:
    X_sub = X.____(columns=drop)                         # ① 열 제거
    rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    r2 = cross_val_score(rf, X_sub, y, cv=3, scoring="r2", n_jobs=-1).mean()
    print(f"{name:<12s} R² = {r2:.4f}")

<details><summary>▶ 정답 보기</summary>

① `drop`

한쪽만 제거하면 R²가 거의 안 변하고, 둘 다 제거할 때만 떨어진다 — 사촌 변수의 대체 가능성.
</details>

---

## 7장 Ames에서 모든 모델 한눈에 비교

단일 트리(약 0.75)는 흔들림이 커 선형회귀(약 0.80)보다 못하지만, 100그루 랜덤 포레스트는 약 0.88로 크게 점프한다. 300그루로 늘려도 거의 같다 — 100~200그루가 가성비 황금률.

### Ping 8 — 네 모델 한 표로 비교

In [ ]:
from sklearn.linear_model import LinearRegression

models = {
    "선형회귀":          LinearRegression(),
    "트리 1그루":        DecisionTreeRegressor(random_state=42),
    "트리 가지치기":     DecisionTreeRegressor(max_depth=8, min_samples_leaf=10, random_state=42),
    "랜덤 포레스트 100": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
}
for name, m in models.items():
    r2 = cross_val_score(m, X, y, cv=5, scoring="r2", n_jobs=-1).mean()
    print(f"{name:<16s} R² = {r2:.4f}")

### Pong 8 — 300그루를 추가해 100그루와 비교

In [ ]:
for nt in [100, 300]:
    rf = RandomForestRegressor(n_estimators=____, random_state=42, n_jobs=-1)  # ① nt
    r2 = cross_val_score(rf, X, y, cv=5, scoring="r2", n_jobs=-1).mean()
    print(f"{nt}그루: R² = {r2:.4f}")

<details><summary>▶ 정답 보기</summary>

① `nt`

300그루는 100그루보다 R²가 거의 같다(차이 약 0.0004) — 시간만 3배.
</details>

---

## 8장 랜덤 포레스트의 한계 — 부스팅으로 가는 다리

랜덤 포레스트는 분산만 줄이고 편향은 거의 못 줄여 Ames에서 약 R² 0.88에 머문다. 부스팅은 정반대로, 트리들이 한 줄로 서서 앞 트리가 못 잡은 잔차를 다음 트리가 메우며 편향을 깎는다.

![편향-분산 과녁]({FIG}/08_bias_variance.png)

### Ping 9 — 랜덤 포레스트의 성능 상한 확인

In [ ]:
for nt in [100, 300, 500]:
    rf = RandomForestRegressor(n_estimators=nt, random_state=42, n_jobs=-1)
    r2 = cross_val_score(rf, X, y, cv=3, scoring="r2", n_jobs=-1).mean()
    print(f"{nt}그루: R² = {r2:.4f}")   # 그루를 늘려도 상한(편향)에서 멈춘다

### Pong 9 — 부스팅의 핵심 직관(잔차를 한 그루씩 메우기)

In [ ]:
M, lr = 50, 0.1
pred = np.full(len(y), y.____())          # ① 첫 예측 = 전체 평균
for m in range(M):
    residual = y.values - pred            # 아직 못 맞힌 양(잔차)
    t = DecisionTreeRegressor(max_depth=3, random_state=m)
    t.fit(X, ____)                        # ② 잔차를 학습
    pred = pred + lr * t.predict(X)
from sklearn.metrics import r2_score
print("부스팅 직관 R²:", round(r2_score(y, pred), 4))

<details><summary>▶ 정답 보기</summary>

① `mean` &nbsp;&nbsp; ② `residual`

첫 트리는 평균을 예측하고, 다음 트리들이 잔차를 메우며 편향을 깎는다. 이 발상이 4부 AdaBoost·GBM으로 이어진다.
</details>

---

## 마무리

평균은 분산을 줄이지만(`배깅`·랜덤 포레스트), 트리들이 닮으면 한계에 부딪힌다. `max_features`로 닮음을 깨고, OOB로 공짜 검증을 얻으며, MDI·순열 중요도로 변수를 두 관점에서 본다. 랜덤 포레스트가 못 줄이는 편향은 다음 부의 **부스팅**이 맡는다.